## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [15]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
# load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
mariamwael4@gmail.com
www.linkedin.com/in/
mariamokashaa (LinkedIn)
Top Skills
Critical Thinking
Planning & Scheduling
Communication
Languages
Arabic (Native or Bilingual)
English (Full Professional)
German (Limited Working)
Certifications
Biomedical Engineering Diploma
AWS Certified Machine Learning -
Specialty
Mariam Okasha
Advanced Analytics Engineer at ITWorx
Cairo, Egypt
Experience
ITWorx
Advanced Analytics Engineer
September 2025 - Present (7 months)
EGMED
Product Specialist
March 2024 - September 2024 (7 months)
The General Authority for Healthcare Accreditation and Regulation
Medical Planning Intern
September 2023 - February 2024 (6 months)
Rofayda Health Park 
Biomedical Engineering Intern
August 2023 - August 2023 (1 month)
INFOSYSTA- Digital Transformation experts in Middle East
Junior Jira Consultant
August 2022 - September 2022 (2 months)
Children's Cancer Hospital Foundation 57357
2 months
Health Information System Intern
October 2021 - October 2021 (1 month)


In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Mariam Okasha"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Mariam Okasha. You are answering questions on Mariam Okasha's website, particularly questions related to Mariam Okasha's career, background, skills and experience. Your responsibility is to represent Mariam Okasha for interactions on the website as faithfully as possible. You are given a summary of Mariam Okasha's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Mariam Okasha. I’m a biomedical engineer turned Advanced Analytics Engineer, with a strong focus on machine learning, deep learning, and building practical AI systems. I studied biomedical engineering at Cairo University and later completed the ITI program, where I deepened my skills in data science, MLOps, and AI engineering. I’m based in Egypt and currently building my career at the intersection of healthcare 

In [ ]:
def chat(question, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]
    response = openai.chat.completions.create(
        model="gpt-4o-mini", 
        messages=messages
    )
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [16]:
import os
load_dotenv(override=True)
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [31]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(
        model="gemini-2.5-flash", 
        messages=messages, 
        response_format=Evaluation
    )
    return response.choices[0].message.parsed

In [18]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(
    model="gpt-4o-mini", 
    messages=messages
)
reply = response.choices[0].message.content

In [19]:
reply

'I currently do not hold a patent. My focus has primarily been on building practical AI systems and working on projects like Medica, rather than pursuing patents at this stage of my career. However, I am always open to innovative ideas and collaborations that could potentially lead to patentable work in the future! If you have any specific projects or concepts in mind, feel free to share.'

In [32]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent correctly states that Mariam Okasha does not hold a patent, as this information is not present in the provided context. The response is professional, engaging, and aligns with the persona's goal by elaborating on current focuses and inviting further discussion about future collaborations, which is suitable for a potential client or employer.")

In [34]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [35]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [36]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The agent's response is completely unacceptable. Responding in Pig Latin is unprofessional, unengaging, and inappropriate for an agent representing Mariam Okasha to potential clients or employers on a website. The agent should directly and professionally answer the question, stating that Mariam does not have any patents, as none are mentioned in the provided context.
Failed evaluation - retrying
The agent responded in Pig Latin, which is highly unprofessional and completely inappropriate for a website representing Mariam Okasha. The response should be clear, direct, and maintain a professional and engaging tone as instructed. It fails to answer the question directly in an understandable way.
